# BirdCLEF+ 2026 — Train & Submit

Single notebook: train EfficientNet-B0 on mel spectrograms → generate submission.csv  
Metric: Macro ROC-AUC (skips classes with no true positives)  
Constraint: CPU ≤ 90 min

In [ ]:
import os, ast, time, warnings, multiprocessing
warnings.filterwarnings('ignore')
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import timm
from tqdm import tqdm

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : cpu')
print(f'CPU cores: {multiprocessing.cpu_count()}')

## 1. Config

In [ ]:
IS_KAGGLE = Path('/kaggle').exists()

# Paths
DATA_DIR     = Path('/kaggle/input/birdclef-2026') if IS_KAGGLE else Path('../data/raw')
WORKING_DIR  = Path('/kaggle/working')             if IS_KAGGLE else Path('../outputs/submission')
MEL_CACHE    = WORKING_DIR / 'mels'
MODEL_PATH   = WORKING_DIR / 'best_model.pth'
SUB_PATH     = WORKING_DIR / 'submission.csv'

WORKING_DIR.mkdir(parents=True, exist_ok=True)
MEL_CACHE.mkdir(parents=True, exist_ok=True)

# Audio
SR          = 32000
DURATION    = 5.0
N_MELS      = 128
N_FFT       = 1024
HOP_LENGTH  = 320
FMIN        = 20.0
FMAX        = 16000.0

# Training
EPOCHS      = 5
BATCH_SIZE  = 64
LR          = 1e-3
WEIGHT_DECAY= 1e-4
NUM_WORKERS = min(4, multiprocessing.cpu_count())
SEED        = 42
VAL_FOLD    = 0
MODEL_NAME  = 'efficientnet_b0'

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Data dir : {DATA_DIR}  exists={DATA_DIR.exists()}')
print(f'Mel cache: {MEL_CACHE}')

## 2. Load Metadata

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_df['primary_label'] = train_df['primary_label'].astype(str)

# Parse secondary labels
train_df['secondary_labels'] = train_df['secondary_labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

species_list = sorted(train_df['primary_label'].unique().tolist())
NUM_CLASSES  = len(species_list)
species2idx  = {s: i for i, s in enumerate(species_list)}

print(f'Total recordings : {len(train_df):,}')
print(f'Species          : {NUM_CLASSES}')
train_df[['primary_label','common_name','rating','filename']].head(3)

## 3. Pre-compute Mel Spectrograms

Uses `soundfile` (40× faster than librosa for native-sr files). Writes `.npy` to disk once.

In [ ]:
def load_and_mel(args):
    filename, audio_dir, cache_dir = args
    cache_path = cache_dir / (filename.replace('/', '__').replace('.ogg', '.npy'))
    if cache_path.exists():
        return filename, True
    try:
        wav, native_sr = sf.read(str(audio_dir / filename), dtype='float32')
        if wav.ndim > 1:
            wav = wav[:, 0]
        if native_sr != SR:
            wav = librosa.resample(wav, orig_sr=native_sr, target_sr=SR)
        # Random-length crop to DURATION
        target_len = int(SR * DURATION)
        if len(wav) <= target_len:
            wav = np.pad(wav, (0, target_len - len(wav)))
        else:
            start = np.random.randint(0, len(wav) - target_len)
            wav = wav[start: start + target_len]
        mel = librosa.feature.melspectrogram(
            y=wav, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
            hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        np.save(cache_path, mel_db.astype(np.float32))
        return filename, True
    except Exception:
        return filename, False


t0 = time.time()
audio_dir = DATA_DIR / 'train_audio'
args_list = [(row['filename'], audio_dir, MEL_CACHE) for _, row in train_df.iterrows()]

already = sum(1 for _ in MEL_CACHE.glob('*.npy'))
print(f'Already cached: {already} / {len(train_df)}')

if already < len(train_df):
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futures = {ex.submit(load_and_mel, a): a[0] for a in args_list}
        for f in tqdm(as_completed(futures), total=len(futures), desc='Mel precompute'):
            pass

total_cached = sum(1 for _ in MEL_CACHE.glob('*.npy'))
print(f'Cached {total_cached} mels in {(time.time()-t0)/60:.1f} min')

## 4. Dataset & DataLoaders

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, cache_dir, train=True):
        self.df = df.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = self.cache_dir / (row['filename'].replace('/', '__').replace('.ogg', '.npy'))
        mel = np.load(cache_path)

        # During training: random horizontal shift (time augmentation)
        if self.train:
            shift = np.random.randint(0, mel.shape[1])
            mel = np.roll(mel, shift, axis=1)

        mel_t = torch.from_numpy(mel).unsqueeze(0)  # (1, n_mels, time)

        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        if row['primary_label'] in species2idx:
            label[species2idx[row['primary_label']]] = 1.0
        for s in row.get('secondary_labels', []):
            if s in species2idx:
                label[species2idx[s]] = 1.0
        return mel_t, label


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
splits = list(skf.split(train_df, train_df['primary_label']))
train_idx, val_idx = splits[VAL_FOLD]

train_ds = BirdDataset(train_df.iloc[train_idx], MEL_CACHE, train=True)
val_ds   = BirdDataset(train_df.iloc[val_idx],   MEL_CACHE, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 5. Model

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, num_classes, model_name='efficientnet_b0', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, in_chans=1, num_classes=0)
        self.classifier = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x):
        return self.classifier(self.backbone(x))


device = torch.device('cpu')
model  = BirdModel(NUM_CLASSES, MODEL_NAME, pretrained=True).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

# Verify forward pass
with torch.no_grad():
    dummy = torch.zeros(2, 1, N_MELS, 501)
    out = model(dummy)
print(f'Output shape    : {out.shape}')

## 6. Train

In [ ]:
def macro_roc_auc(targets, preds):
    scores = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() > 0:
            scores.append(roc_auc_score(targets[:, i], preds[:, i]))
    return float(np.mean(scores)) if scores else 0.0


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc = 0.0
history  = []

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()

    # ── Train ──
    model.train()
    train_loss = 0.0
    for mels, labels in tqdm(train_loader, desc=f'Ep{epoch} train', leave=False):
        optimizer.zero_grad()
        loss = criterion(model(mels), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    val_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for mels, labels in tqdm(val_loader, desc=f'Ep{epoch} val', leave=False):
            logits = model(mels)
            val_loss += criterion(logits, labels).item()
            all_preds.append(torch.sigmoid(logits).numpy())
            all_targets.append(labels.numpy())
    val_loss /= len(val_loader)
    auc = macro_roc_auc(np.concatenate(all_targets), np.concatenate(all_preds))
    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_auc': auc})
    elapsed = time.time() - t_ep
    print(f'Ep {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auc={auc:.4f} | {elapsed:.0f}s')

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'       ↑ best model saved (auc={best_auc:.4f})')

print(f'\nBest val AUC: {best_auc:.4f}')

## 7. Inference on Test Soundscapes

In [ ]:
# Load best weights
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

test_dir  = DATA_DIR / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

def predict_soundscape(audio_path):
    wav, native_sr = sf.read(str(audio_path), dtype='float32')
    if wav.ndim > 1:
        wav = wav[:, 0]
    if native_sr != SR:
        wav = librosa.resample(wav, orig_sr=native_sr, target_sr=SR)

    target_len = int(SR * DURATION)
    step_len   = target_len  # non-overlapping 5s windows
    windows, end_secs = [], []
    start = 0
    while start + target_len <= len(wav):
        windows.append(wav[start: start + target_len])
        end_secs.append(int((start + target_len) / SR))
        start += step_len
    if not windows:
        windows.append(np.pad(wav, (0, target_len - len(wav))))
        end_secs.append(5)

    rows = []
    # Batch windows together for speed
    batch_size = 16
    all_probs  = []
    for i in range(0, len(windows), batch_size):
        batch = windows[i: i + batch_size]
        mels  = []
        for w in batch:
            mel = librosa.feature.melspectrogram(
                y=w, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
            mels.append(mel_db.astype(np.float32))
        tensor = torch.from_numpy(np.stack(mels)).unsqueeze(1)
        with torch.no_grad():
            probs = torch.sigmoid(model(tensor)).numpy()
        all_probs.append(probs)

    all_probs = np.concatenate(all_probs)
    stem = audio_path.stem
    for probs, end_sec in zip(all_probs, end_secs):
        row = {'row_id': f'{stem}_{end_sec}'}
        for j, sp in enumerate(species_list):
            row[sp] = probs[j]
        rows.append(row)
    return rows


all_rows = []
for fp in tqdm(test_files, desc='Inference'):
    all_rows.extend(predict_soundscape(fp))

submission = pd.DataFrame(all_rows)
submission.to_csv(SUB_PATH, index=False)
print(f'Saved {len(submission):,} rows → {SUB_PATH}')
submission.head(3)

## 8. Validate Submission

In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
expected_cols = set(sample_sub.columns)
actual_cols   = set(submission.columns)

missing = expected_cols - actual_cols
extra   = actual_cols   - expected_cols

print(f'Submission shape : {submission.shape}')
print(f'Missing cols     : {missing if missing else "none"}')
print(f'Extra cols       : {extra   if extra   else "none"}')
print(f'All scores in [0,1]: {((submission[species_list] >= 0) & (submission[species_list] <= 1)).all().all()}')
print('\nDone!')